# Pango events within the ARG

In this notebook we examine how well the ARG reflects evolutionary events implicit in the pango naming system.


## Summary

Of the 2058 distinct pango lineages in the ARG, 1473 of these (comprising 717798 samples) match perfectly, with unique origination events in the ARG where all samples assigned a given lineage descend from the first node assigned that lineage. A further 245 lineages (589197 samples) match perfectly when we count the descendants of the  parent of the first node (accounting for polytomies in which multiple originating nodes for a given lineage are siblings). We then have 306 lineages (755840 samples) where the difference in the number descendants of the first node's parent is < 100. The remaining 35 lineages (420047 samples) are dominated by a few large lineages such as BA.1.1 (147271 samples) and AY.4.2 (54607 samples) which have multiple non-sibling origins within the ARG.


In [1]:
import sc2ts
import tszip
import pathlib
import numpy as np
import pandas as pd
import concurrent.futures as cf
from tqdm.notebook import tqdm

datadir = pathlib.Path("../data")

## Code

In [2]:
ts = tszip.load(datadir / "sc2ts_viridian_v2.0.trees.tsz")

In [3]:
df_node = sc2ts.node_data(ts, inheritance_stats=True).set_index("node_id")
df_node["node_time"] = ts.nodes_time
df_node

,pango,sample_id,scorpio,is_sample,is_recombinant,num_mutations,max_descendant_samples,date,node_time
node_id,,,,,,,,,
0,B,Vestigial_ignore,.,False,False,0,0,2019-11-19,1661.236253
1,B,Wuhan/Hu-1/2019,.,False,False,0,3128669,2019-12-26,1624.000000
2,A,SRR11772659,.,True,False,1,320,2020-01-19,1600.000000
3,B,SRR11397727,.,True,False,0,1,2020-01-24,1595.000000
4,B,SRR11397730,.,True,False,0,1,2020-01-24,1595.000000
...,...,...,...,...,...,...,...,...,...
3424141,HV.1.1,,Omicron (XBB.1-like),False,False,3,18,2023-09-09,271.928289
3424142,BQ.1.1.29,,Omicron (BA.5-like),False,False,9,158,2022-10-02,613.645718
3424143,BA.1.1,,Omicron (BA.1-like),False,False,1,27,2021-11-02,947.672921


## Concordance with original assignments by Viridian

We extract the original Pango lineage assignments (version 1.29 and version 1.21) from the dataset and join this with our nodes samples to measure the levels of concordance between the three different calls for the same sequences.

In [4]:
ds = sc2ts.Dataset(datadir / "viridian_mafft_2024-10-14_v1.vcz.zip")
ds

Dataset at ../data/viridian_mafft_2024-10-14_v1.vcz.zip with 4484157 samples, 29903 variants, and 30 metadata fields. See ds.metadata.field_descriptors() for a description of the fields.

In [5]:
df_sample = df_node[df_node.is_sample].set_index("sample_id")
df_sample

,pango,scorpio,is_sample,is_recombinant,num_mutations,max_descendant_samples,date,node_time
sample_id,,,,,,,,
SRR11772659,A,.,True,False,1,320,2020-01-19,1600.0
SRR11397727,B,.,True,False,0,1,2020-01-24,1595.0
SRR11397730,B,.,True,False,0,1,2020-01-24,1595.0
SRR11597198,A,.,True,False,0,1,2020-01-25,1594.0
SRR11597221,A,.,True,False,0,1,2020-01-25,1594.0
...,...,...,...,...,...,...,...,...
SRR29413084,KP.2.15,Omicron (BA.2-like),True,False,0,1,2024-06-02,4.0
SRR29412872,KP.3.3,Omicron (BA.2-like),True,False,0,1,2024-06-04,2.0
SRR29413186,KP.3.1.1,Omicron (BA.2-like),True,False,0,1,2024-06-04,2.0


In [6]:
df_sample = df_sample.join(ds.metadata.as_dataframe(["Viridian_pangolin", "Viridian_pangolin_1.29", "Viridian_scorpio_1.29"]))

In [7]:
df_sample = df_sample.rename(columns={"Viridian_pangolin": "Viridian_pangolin_1.21"})
df_sample

,pango,scorpio,is_sample,is_recombinant,num_mutations,max_descendant_samples,date,node_time,Viridian_pangolin_1.21,Viridian_pangolin_1.29,Viridian_scorpio_1.29
sample_id,,,,,,,,,,,
SRR11772659,A,.,True,False,1,320,2020-01-19,1600.0,A,A,.
SRR11397727,B,.,True,False,0,1,2020-01-24,1595.0,B,B,.
SRR11397730,B,.,True,False,0,1,2020-01-24,1595.0,B,B,.
SRR11597198,A,.,True,False,0,1,2020-01-25,1594.0,A,A,.
SRR11597221,A,.,True,False,0,1,2020-01-25,1594.0,A,A,.
...,...,...,...,...,...,...,...,...,...,...,...
SRR29413084,KP.2.15,Omicron (BA.2-like),True,False,0,1,2024-06-02,4.0,BA.2,KP.2.15,Omicron (BA.2-like)
SRR29412872,KP.3.3,Omicron (BA.2-like),True,False,0,1,2024-06-04,2.0,BA.2,KP.3.3,Omicron (BA.2-like)
SRR29413186,KP.3.1.1,Omicron (BA.2-like),True,False,0,1,2024-06-04,2.0,BA.2,KP.3.1.1,Omicron (BA.2-like)


In [8]:
np.sum(df_sample["Viridian_pangolin_1.21"] != df_sample["Viridian_pangolin_1.29"])

np.int64(74410)

In [9]:
print(f"{_ / df_sample.shape[0] * 100: .2f}")

 2.38


In [10]:
np.sum(df_sample["pango"] != df_sample["Viridian_pangolin_1.29"])

np.int64(28616)

In [11]:
print(f"{_ / df_sample.shape[0] * 100: .2f}")

 0.91


Only 712 samples differ in their Scorpio designations.

In [12]:
df_sample[df_sample["scorpio"] != df_sample["Viridian_scorpio_1.29"]][["scorpio", "Viridian_scorpio_1.29"]]

,scorpio,Viridian_scorpio_1.29
sample_id,,
SRR27324949,.,Beta (B.1.351-like)
SRR27237896,.,Beta (B.1.351-like)
SRR13620105,.,Beta (B.1.351-like)
SRR13620108,.,Beta (B.1.351-like)
SRR13620095,.,Beta (B.1.351-like)
...,...,...
ERR13322696,Omicron (BA.2-like),Probable Omicron (BA.2-like)
SRR29261519,Omicron (BA.2-like),Omicron (Unassigned)
SRR28516178,Omicron (XBB-like),Omicron (XBB.1-like)


## Origination events

How well does the ARG reflect the phylogenetic structure of the pango naming system?

We sort the nodes by the descending samples first, and then by node time. This should guarantee that the first node in the dataframe for each pango lineage is the "majority" node for that pango.

In [13]:
dfn_sorted = df_node.sort_values(["max_descendant_samples", "node_time"], ascending=False)
dfn_sorted

,pango,sample_id,scorpio,is_sample,is_recombinant,num_mutations,max_descendant_samples,date,node_time
node_id,,,,,,,,,
1,B,Wuhan/Hu-1/2019,.,False,False,0,3128669,2019-12-26,1624.000000
33,B,,.,False,False,1,3123318,2019-12-26,1624.000000
12,B.1,SRR11597205,.,True,False,2,3123314,2020-01-28,1591.515332
67,B.1,,.,False,False,1,3123299,2020-01-28,1591.515332
118,B.1.1,,.,False,False,3,1664388,2020-01-28,1591.004716
...,...,...,...,...,...,...,...,...,...
1896623,JN.1.4.5,SRR29413085,Omicron (BA.2-like),True,False,6,1,2024-06-06,0.000000
1896624,LB.1.7,SRR29413113,Omicron (BA.2-like),True,False,2,1,2024-06-06,0.000000
1896625,KP.1.1.3,SRR29413120,Omicron (BA.2-like),True,False,7,1,2024-06-06,0.000000


In [14]:
dfn_pango = dfn_sorted.reset_index().groupby(["pango"]).first()
dfn_pango

,node_id,sample_id,scorpio,is_sample,is_recombinant,num_mutations,max_descendant_samples,date,node_time
pango,,,,,,,,,
A,9,,.,False,False,1,1323,2019-12-26,1624.000000
A.1,279,,.,False,False,2,283,2020-01-19,1600.000000
A.11,4099,SRR23586010,.,True,False,3,1,2020-03-28,1531.000000
A.2,663,,.,False,False,4,242,2020-02-01,1587.000000
A.2.2,1163,ERR12537611,.,True,False,1,81,2020-03-19,1540.000000
...,...,...,...,...,...,...,...,...,...
XW,1378600,,Omicron (BA.2-like),False,True,1,40,2022-03-03,826.540427
XY,1417358,,Omicron (Unassigned),False,True,2,23,2022-03-15,814.000000
XZ,1383518,,Omicron (BA.2-like),False,False,1,51,2022-03-09,820.000000


In [15]:

def worker(work):
    pango, row = work
    df_pango = df_node[df_node.pango == pango]
    samples = df_pango[df_pango.is_sample].index
    root = row["node_id"]
    tracked_samples = []
    parent_tracked_samples = []
    for tree in ts.trees(tracked_samples=samples):
        tracked_samples.append(tree.num_tracked_samples(root))
        parent = tree.parent(root)
        if parent != -1:
            parent_tracked_samples.append(tree.num_tracked_samples(parent))
        else:
            parent_tracked_samples.append(0)
            
    return {
        "pango": pango,
        "root": root,
        "total_samples": len(samples),
        "max_descendants": np.max(tracked_samples),
        "min_descendants": np.min(tracked_samples),
        "parent_max_descendants": np.max(parent_tracked_samples),
        "parent_min_descendants": np.min(parent_tracked_samples),
    }
    
# Note: set things up this way with an eye to using concurrent.futures,
# but it was totally GIL-blocked, seemingly. Not worth setting up
# process level parallelism.
data = []
for work in tqdm(dfn_pango.iterrows(), total=dfn_pango.shape[0]):
    result = worker(work)
    data.append(result)
        
df_pango_events = pd.DataFrame(data)
df_pango_events

  0%|          | 0/3029 [00:00<?, ?it/s]

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants
0,A,9,317,317,317,317,317
1,A.1,279,283,283,283,283,283
2,A.11,4099,1,1,1,1,1
3,A.2,663,56,56,56,56,56
4,A.2.2,1163,81,81,81,81,81
...,...,...,...,...,...,...,...
3024,XW,1378600,37,37,37,37,37
3025,XY,1417358,23,23,23,23,23
3026,XZ,1383518,51,51,51,51,51
3027,Y.1,60911,38,38,38,38,38


In [16]:
total = df_pango_events["total_samples"]
diff = (total - df_pango_events["max_descendants"]).abs() 
diff_parent = (total - df_pango_events["parent_max_descendants"]).abs() 
df_pango_events["diff"] = diff
df_pango_events["diff_parent"] = diff_parent
df_pango_events["relative_diff"] = diff / total
df_pango_events["relative_diff_parent"] = diff_parent / total

In [17]:
df_pango_events

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants,diff,diff_parent,relative_diff,relative_diff_parent
0,A,9,317,317,317,317,317,0,0,0.0,0.0
1,A.1,279,283,283,283,283,283,0,0,0.0,0.0
2,A.11,4099,1,1,1,1,1,0,0,0.0,0.0
3,A.2,663,56,56,56,56,56,0,0,0.0,0.0
4,A.2.2,1163,81,81,81,81,81,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
3024,XW,1378600,37,37,37,37,37,0,0,0.0,0.0
3025,XY,1417358,23,23,23,23,23,0,0,0.0,0.0
3026,XZ,1383518,51,51,51,51,51,0,0,0.0,0.0
3027,Y.1,60911,38,38,38,38,38,0,0,0.0,0.0


In [18]:
perfect = df_pango_events[df_pango_events["diff"] == 0]
perfect

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants,diff,diff_parent,relative_diff,relative_diff_parent
0,A,9,317,317,317,317,317,0,0,0.0,0.0
1,A.1,279,283,283,283,283,283,0,0,0.0,0.0
2,A.11,4099,1,1,1,1,1,0,0,0.0,0.0
3,A.2,663,56,56,56,56,56,0,0,0.0,0.0
4,A.2.2,1163,81,81,81,81,81,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
3024,XW,1378600,37,37,37,37,37,0,0,0.0,0.0
3025,XY,1417358,23,23,23,23,23,0,0,0.0,0.0
3026,XZ,1383518,51,51,51,51,51,0,0,0.0,0.0
3027,Y.1,60911,38,38,38,38,38,0,0,0.0,0.0


In [19]:
perfect.shape

(2033, 11)

In [20]:
perfect.total_samples.sum()

np.int64(912988)

# Consider the effects of polytomies 

In [21]:
perfect_for_parent = df_pango_events[(df_pango_events["diff"] > 0) & (df_pango_events["diff_parent"] == 0)]
perfect_for_parent

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants,diff,diff_parent,relative_diff,relative_diff_parent
14,A.5,934,57,56,56,57,57,1,0,0.017544,0.0
37,AY.1,261685,476,468,468,476,476,8,0,0.016807,0.0
42,AY.103,290372,74947,74935,74928,74947,74942,12,0,0.000160,0.0
45,AY.105,288646,242,175,173,242,242,67,0,0.276860,0.0
58,AY.116,331107,176,96,95,176,176,80,0,0.454545,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2918,XBB.2.4,1760525,29,28,28,29,29,1,0,0.034483,0.0
2924,XBB.2.7,1751662,9,3,3,9,9,6,0,0.666667,0.0
2946,XBF.2,1759867,7,4,4,7,7,3,0,0.428571,0.0
2949,XBF.5,1773800,8,7,7,8,8,1,0,0.125000,0.0


In [22]:
perfect_for_parent.shape

(416, 11)

In [23]:
perfect_for_parent.total_samples.sum()

np.int64(708186)

# The rest

In [24]:
not_perfect = df_pango_events[df_pango_events["diff_parent"] != 0]
not_perfect

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants,diff,diff_parent,relative_diff,relative_diff_parent
39,AY.100,308223,21263,20177,19874,20177,19874,1086,1086,0.051075,0.051075
41,AY.102,830925,7,5,5,5,5,2,2,0.285714,0.285714
46,AY.106,286854,171,168,168,170,170,3,1,0.017544,0.005848
47,AY.107,257750,839,633,630,633,630,206,206,0.245530,0.245530
49,AY.109,390866,486,476,475,476,475,10,10,0.020576,0.020576
...,...,...,...,...,...,...,...,...,...,...,...
3002,XDP,1871948,80,62,62,62,62,18,18,0.225000,0.225000
3003,XDP.1,1883376,32,24,24,24,24,8,8,0.250000,0.250000
3005,XDP.3,1875065,15,13,13,13,13,2,2,0.133333,0.133333
3010,XEB,1876701,9,5,5,5,5,4,4,0.444444,0.444444


In [25]:
not_perfect.total_samples.sum()

np.int64(1508297)

## Lineages that are pretty close 

And have a reasonable number of samples

In [26]:
close_to_right = not_perfect[(not_perfect["diff_parent"] < 100)]
close_to_right

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants,diff,diff_parent,relative_diff,relative_diff_parent
41,AY.102,830925,7,5,5,5,5,2,2,0.285714,0.285714
46,AY.106,286854,171,168,168,170,170,3,1,0.017544,0.005848
49,AY.109,390866,486,476,475,476,475,10,10,0.020576,0.020576
51,AY.110,327712,802,746,746,746,746,56,56,0.069825,0.069825
52,AY.111,252163,5206,5159,5159,5199,5199,47,7,0.009028,0.001345
...,...,...,...,...,...,...,...,...,...,...,...
3002,XDP,1871948,80,62,62,62,62,18,18,0.225000,0.225000
3003,XDP.1,1883376,32,24,24,24,24,8,8,0.250000,0.250000
3005,XDP.3,1875065,15,13,13,13,13,2,2,0.133333,0.133333
3010,XEB,1876701,9,5,5,5,5,4,4,0.444444,0.444444


In [27]:
close_to_right.shape

(538, 11)

In [28]:
close_to_right.total_samples.sum()

np.int64(504972)

## Important stuff that's a long way off

In [29]:
important = not_perfect[not_perfect["diff_parent"] >= 100]
important.sort_values("total_samples", ascending=False)

,pango,root,total_samples,max_descendants,min_descendants,parent_max_descendants,parent_min_descendants,diff,diff_parent,relative_diff,relative_diff_parent
136,AY.4,258747,552689,550009,549746,551862,551605,2680,827,0.004849,0.001496
983,BA.1.1,906313,177589,156705,156565,176646,176602,20884,943,0.117597,0.005310
1017,BA.1.17.2,953855,49496,33211,33198,33213,33200,16285,16283,0.329016,0.328976
1158,BA.2.9,990501,37866,34837,34832,37632,37628,3029,234,0.079993,0.006180
1009,BA.1.15,927101,27919,24652,24643,24658,24649,3267,3261,0.117017,0.116802
39,AY.100,308223,21263,20177,19874,20177,19874,1086,1086,0.051075,0.051075
993,BA.1.1.18,998365,19370,9604,9588,10251,10234,9766,9119,0.504182,0.470780
1018,BA.1.18,960653,13307,8863,8847,8865,8849,4444,4442,0.333960,0.333809
131,AY.39,273098,12971,12098,11952,12098,11952,873,873,0.067304,0.067304
1033,BA.2.10,1211258,11234,12,12,12,12,11222,11222,0.998932,0.998932


In [30]:
important.shape

(43, 11)

In [31]:
important.total_samples.sum()

np.int64(1003325)

In [32]:
df_pango_events.to_csv(datadir / "pango_events_in_arg.csv", index=False)